# Event displays

Look at simulated events with LUCiD's visualization tools. First one SK-like muon **three ways**
(2D unrolled map, time-evolution animation, interactive 3D discs), then the right display for
**other geometries and materials**. Each geometry uses the display that actually fits it:
oriented **discs** for wall-mounted PMTs (cylinder/sphere/box) and a **3D scatter** for the
point DOMs of a neutrino-telescope string. All are library calls; no external data needed.

In [ ]:
import sys; sys.path.append('..')
%matplotlib inline
import jax, numpy as np
import plotly.graph_objects as go
from lucid.geometry import generate_detector
from lucid.simulation import setup_event_simulator
from lucid.detector_params import ParticleParams
from lucid.visualization import create_detector_display, animate_event

GEOM = '../config/SK_like_geom_config.json'
PHYS = '../config/SK_like_physics_config.json'
det = generate_detector(GEOM)
sim = setup_event_simulator(GEOM, 250_000, K=6, hit_mode='realistic', physics_config=PHYS,
                            default_detector_params=True, particle='muon',
                            n_cap=120, n_angular=200, n_height=120)
track = ParticleParams.from_cartesian(energy=1000., position=[0., 0., 0.], direction=[1., 0., 0.], t0=0.)
charge, time = (np.asarray(x) for x in jax.lax.stop_gradient(sim(track, jax.random.PRNGKey(0))))
print(f'{(charge>0).sum()} PMTs lit, total charge {charge.sum():.0f} pe')

def bright(ch, frac=0.05):
    """Indices of the bright sensors (the ring) — keeps disc plots light and informative."""
    return np.where(ch > frac * ch.max())[0]

## 1. SK cylinder — three ways
2D unrolled map, time animation, and the interactive 3D disc view. (The 2D unroll and the
animation are cylinder-specific.)

In [ ]:
display = create_detector_display(GEOM, sparse=False)
display(charge, time, file_name=None, perc_min=0.0, perc_max=99.5)

In [ ]:
from IPython.display import Image
animate_event(det, charge, time, out_path='event.gif', n_frames=24, fps=12)
Image('event.gif')

In [ ]:
idx = bright(charge)
det.visualize_event_data_plotly_discs(idx, charge[idx], time[idx], log_scale=True,
                                      show_all_sensors=False, title='SK cylinder -- muon (bright ring)')

## 2. Other geometries — the display that fits
Only the config files and `detector_type` change. **Discs** for the closed geometries (sphere,
box), a **3D scatter** for the telescope string (DOMs are point sensors with no surface normal,
so oriented discs don't apply).

In [ ]:
def simulate(geom, phys, dtype, energy, direction, position=None, wavelength_mode=False, n_photons=50_000, K=4):
    """Forward muon event; returns (detector, per-sensor charge, per-sensor time)."""
    s = setup_event_simulator(geom, n_photons, K=K, temperature=None, is_calibration=False,
            detector_type=dtype, physics_config=phys, default_detector_params=True,
            hit_mode='aggregated', wavelength_mode=wavelength_mode, particle='muon', use_expected_value=True)
    d = s.det_geom.detector
    pts = np.asarray(d.all_points)
    if position is None:
        position = pts.mean(0).tolist()          # array centre (telescope)
    tr = ParticleParams.from_cartesian(energy=energy, position=position, direction=direction, t0=0.)
    ch, tm = (np.asarray(x) for x in s(tr, jax.random.PRNGKey(0)))
    return d, ch, tm

### Sphere (JUNO / water) and box (MidBox / water) — oriented discs

In [ ]:
for title, g, p, dt in [('sphere / water (JUNO)', 'JUNO', 'JUNO', 'sphere'),
                        ('box / water (MidBox)',  'MidBox', 'MidBox', 'box')]:
    d, ch, tm = simulate(f'../config/{g}_geom_config.json', f'../config/{p}_physics_config.json', dt, 1000., [1,0,0], [0,0,0])
    idx = bright(ch)
    print(f'{title}: {len(idx)} bright / {int((ch>0).sum())} lit')
    d.visualize_event_data_plotly_discs(idx, ch[idx], tm[idx], log_scale=True,
                                        show_all_sensors=False, title=title)

### String (IceCube / ice) — 3D scatter of lit DOMs
DOMs are point sensors, so we scatter the lit DOMs (coloured by charge) over a faint cloud of
all DOMs — the correct display for a telescope (as in `viewer/string/`). A ~100 GeV horizontal
muon through the array centre lights a track of DOMs.

In [ ]:
d, ch, tm = simulate('../config/IceCube86_full_geom_config.json', '../config/IceCube86_ice_physics_config.json',
                     'string', 100_000., [1, 0, 0], None, n_photons=500_000, K=15)  # ice scatters strongly -> K=15
pts = np.asarray(d.all_points); lit = ch > 0
print(f'string / ice (IceCube): {int(lit.sum())} / {len(pts)} DOMs lit')
fig = go.Figure()
fig.add_trace(go.Scatter3d(x=pts[:,0], y=pts[:,1], z=pts[:,2], mode='markers',
              marker=dict(size=1.2, color='gray', opacity=0.12), name='all DOMs', hoverinfo='skip'))
fig.add_trace(go.Scatter3d(x=pts[lit,0], y=pts[lit,1], z=pts[lit,2], mode='markers',
              marker=dict(size=4, color=ch[lit], colorscale='viridis',
                          colorbar=dict(title='charge'), showscale=True), name='lit DOMs'))
fig.update_layout(title='string / ice (IceCube) -- lit DOMs', template='plotly_dark',
                  scene=dict(aspectmode='data'), height=600, margin=dict(l=0, r=0, t=30, b=0))
fig.show()

### Same detector, different material: water vs WbLS
Water-based liquid scintillator adds (near-)isotropic scintillation on top of Cherenkov
(needs `wavelength_mode=True`). Same SK-like tank, same 1 GeV muon.

In [ ]:
_, ch_water, _ = simulate('../config/SK_like_geom_config.json',      '../config/SK_like_physics_config.json',      'cylinder', 1000., [1,0,0], [0,0,0])
_, ch_wbls,  _ = simulate('../config/SK_like_wbls_geom_config.json', '../config/SK_like_wbls_physics_config.json', 'cylinder', 1000., [1,0,0], [0,0,0], wavelength_mode=True)
print(f'water: {int((ch_water>0).sum()):5d} lit, total {ch_water.sum():8.0f} pe')
print(f'WbLS : {int((ch_wbls>0).sum()):5d} lit, total {ch_wbls.sum():8.0f} pe  (scintillation adds light)')

**Correct display per geometry:** 2D-unroll + animation are cylinder-only; oriented **discs**
work for cylinder/sphere/box (they have surface normals); **string** DOMs have no surface normal,
so a 3D **scatter** is the right view. The ice config reuses the water SIREN emitter for now.

**Next:** `00_quickstart` to build a simulator, or `examples/hello_telescope.py` for the
string + cascade workflow.